# LeetCode #59 — Spiral Matrix II: điền ma trận theo xoắn ốc

**Mức độ:** Medium · **Chủ đề:** Array, Matrix, Simulation

Skeleton trên LeetCode:

```python
class Solution:
    def generateMatrix(self, n: int) -> List[List[int]]:
```

Bài #54 **đọc** ma trận theo xoắn ốc; bài này **ghi**: điền các số `1, 2, …, n²` vào ma trận `n × n` theo đúng đường xoắn ốc.

Mục tiêu của notebook:

1. Tạo ma trận 2 chiều đúng cách (tránh bẫy `[[0] * n] * n`).
2. Kỹ thuật **bốn biên** `top, bottom, left, right` co dần.
3. Kỹ thuật **bảng hướng + rẽ phải khi gặp tường**.
4. Tính trực tiếp giá trị ở một ô **không cần dựng ma trận** (thử thách cuối).

# 1. Bài toán là gì?

Cho số nguyên dương `n`. Tạo ma trận `n × n` chứa các số từ `1` tới `n²` theo **thứ tự xoắn ốc**: bắt đầu ở góc trên trái, đi phải → xuống → trái → lên, rồi lặp vào trong.

**Constraints (theo đề):** `1 <= n <= 20`

---

## Ví dụ

```text
n = 3  ->  [[1, 2, 3],
            [8, 9, 4],
            [7, 6, 5]]

n = 1  ->  [[1]]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Ma trận (mảng hai chiều) trong Python

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [1]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [2]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [3]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [4]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.2 Áp dụng vào bài này

Ngược với #54: tạo **ma trận** `n × n` rỗng (bằng vòng `for`, không `[[0]*n]*n`) rồi đi xoắn ốc và điền `1, 2, …, n²` bằng bốn biên hoặc **mảng hướng**.

# 3. Nền tảng: tạo ma trận 2 chiều

`[[0] * n for _ in range(n)]` tạo `n` hàng **độc lập**. Còn `[[0] * n] * n` tạo `n` tham chiếu tới **cùng một hàng** — sửa một ô là sửa cả cột!

In [5]:
from typing import List

good = [[0] * 3 for _ in range(3)]
bad = [[0] * 3] * 3
good[0][0] = 9
bad[0][0] = 9
print("Đúng:", good)
print("Sai :", bad)

Đúng: [[9, 0, 0], [0, 0, 0], [0, 0, 0]]
Sai : [[9, 0, 0], [9, 0, 0], [9, 0, 0]]


In [6]:
def show(m):
    w = max(len(str(x)) for row in m for x in row)
    for row in m:
        print(" ".join(str(x).rjust(w) for x in row))

# 4. Cách 1: bốn biên

Giữ biên của vùng **chưa điền**: `top = 0, bottom = n-1, left = 0, right = n-1`, và biến `num` là số tiếp theo cần điền.

Mỗi vòng điền một lớp vỏ:

1. Hàng `top`, từ `left` → `right`; rồi `top += 1`.
2. Cột `right`, từ `top` → `bottom`; rồi `right -= 1`.
3. Hàng `bottom`, từ `right` → `left`; rồi `bottom -= 1`.
4. Cột `left`, từ `bottom` → `top`; rồi `left += 1`.

Với ma trận **vuông**, các kiểm tra `if top <= bottom` / `if left <= right` (bắt buộc ở bài #54 cho ma trận chữ nhật) vẫn nên giữ để code an toàn và dùng lại được.

In [7]:
def generate_matrix(n: int) -> List[List[int]]:
    mat = [[0] * n for _ in range(n)]
    top, bottom, left, right = 0, n - 1, 0, n - 1
    num = 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):          # phải
            mat[top][c] = num; num += 1
        top += 1
        for r in range(top, bottom + 1):          # xuống
            mat[r][right] = num; num += 1
        right -= 1
        if top <= bottom:
            for c in range(right, left - 1, -1):  # trái
                mat[bottom][c] = num; num += 1
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1):  # lên
                mat[r][left] = num; num += 1
            left += 1
    return mat

show(generate_matrix(3))
print()
show(generate_matrix(4))

1 2 3
8 9 4
7 6 5

 1  2  3  4
12 13 14  5
11 16 15  6
10  9  8  7


# 5. Mô phỏng từng bước (n = 4)

In [8]:
def rng(a, b):
    return f"{a}..{b}" if b >= a else "(không có ô nào)"


def trace_generate(n: int) -> List[List[int]]:
    mat = [[0] * n for _ in range(n)]
    top, bottom, left, right = 0, n - 1, 0, n - 1
    num = 1
    layer = 0
    while top <= bottom and left <= right:
        print(f"--- lớp {layer}: top={top} bottom={bottom} left={left} right={right}")
        start = num
        for c in range(left, right + 1):
            mat[top][c] = num; num += 1
        top += 1
        print(f"  phải : điền {rng(start, num - 1)} vào hàng {top - 1}")
        start = num
        for r in range(top, bottom + 1):
            mat[r][right] = num; num += 1
        right -= 1
        print(f"  xuống: điền {rng(start, num - 1)} vào cột {right + 1}")
        if top <= bottom:
            start = num
            for c in range(right, left - 1, -1):
                mat[bottom][c] = num; num += 1
            bottom -= 1
            print(f"  trái : điền {rng(start, num - 1)} vào hàng {bottom + 1}")
        if left <= right:
            start = num
            for r in range(bottom, top - 1, -1):
                mat[r][left] = num; num += 1
            left += 1
            print(f"  lên  : điền {rng(start, num - 1)} vào cột {left - 1}")
        layer += 1
    show(mat)
    return mat

_ = trace_generate(4)

--- lớp 0: top=0 bottom=3 left=0 right=3
  phải : điền 1..4 vào hàng 0
  xuống: điền 5..7 vào cột 3
  trái : điền 8..10 vào hàng 3
  lên  : điền 11..12 vào cột 0
--- lớp 1: top=1 bottom=2 left=1 right=2
  phải : điền 13..14 vào hàng 1
  xuống: điền 15..15 vào cột 2
  trái : điền 16..16 vào hàng 2
  lên  : điền (không có ô nào) vào cột 1
 1  2  3  4
12 13 14  5
11 16 15  6
10  9  8  7


Với `n = 4`: lớp 0 điền `1..12`, lớp 1 điền `13..16`. Với `n` lẻ, lớp trong cùng chỉ có **một ô** (bước "phải" điền nó, các bước khác rỗng).

# 6. Cách 2: bảng hướng + rẽ phải

"Đi thẳng; nếu ô kế tiếp ra ngoài hoặc **đã điền** (khác 0) thì rẽ phải."

Hướng theo thứ tự rẽ phải: `(0,1)` phải, `(1,0)` xuống, `(0,-1)` trái, `(-1,0)` lên.

In [9]:
def generate_matrix_dirs(n: int) -> List[List[int]]:
    mat = [[0] * n for _ in range(n)]
    dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    r = c = d = 0
    for num in range(1, n * n + 1):
        mat[r][c] = num
        nr, nc = r + dirs[d][0], c + dirs[d][1]
        if not (0 <= nr < n and 0 <= nc < n and mat[nr][nc] == 0):
            d = (d + 1) % 4
            nr, nc = r + dirs[d][0], c + dirs[d][1]
        r, c = nr, nc
    return mat

show(generate_matrix_dirs(4))

 1  2  3  4
12 13 14  5
11 16 15  6
10  9  8  7


Ở đây chính ma trận đóng vai "đã thăm" (ô còn `0` là chưa điền) — không cần mảng `seen` riêng.

# 7. Phiên bản đúng format LeetCode

In [10]:
class Solution:
    def generateMatrix(self, n: int) -> List[List[int]]:
        mat = [[0] * n for _ in range(n)]
        top, bottom, left, right = 0, n - 1, 0, n - 1
        num = 1
        while top <= bottom and left <= right:
            for c in range(left, right + 1):
                mat[top][c] = num; num += 1
            top += 1
            for r in range(top, bottom + 1):
                mat[r][right] = num; num += 1
            right -= 1
            if top <= bottom:
                for c in range(right, left - 1, -1):
                    mat[bottom][c] = num; num += 1
                bottom -= 1
            if left <= right:
                for r in range(bottom, top - 1, -1):
                    mat[r][left] = num; num += 1
                left += 1
        return mat

In [11]:
def spiral_read(m):
    """Đọc xoắn ốc (bài #54) để kiểm tra ngược."""
    out = []
    top, bottom, left, right = 0, len(m) - 1, 0, len(m[0]) - 1
    while top <= bottom and left <= right:
        out += [m[top][c] for c in range(left, right + 1)]; top += 1
        out += [m[r][right] for r in range(top, bottom + 1)]; right -= 1
        if top <= bottom:
            out += [m[bottom][c] for c in range(right, left - 1, -1)]; bottom -= 1
        if left <= right:
            out += [m[r][left] for r in range(bottom, top - 1, -1)]; left += 1
    return out


def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<10} PASS  {actual}")


sol = Solution()
check("n = 3", sol.generateMatrix(3), [[1, 2, 3], [8, 9, 4], [7, 6, 5]])
check("n = 1", sol.generateMatrix(1), [[1]])
check("n = 2", sol.generateMatrix(2), [[1, 2], [4, 3]])
for n in range(1, 21):
    m = sol.generateMatrix(n)
    assert m == generate_matrix_dirs(n)
    assert spiral_read(m) == list(range(1, n * n + 1))
print("n = 1..20: hai cách khớp nhau; đọc xoắn ốc ra đúng 1..n²")

n = 3      PASS  [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
n = 1      PASS  [[1]]
n = 2      PASS  [[1, 2], [4, 3]]
n = 1..20: hai cách khớp nhau; đọc xoắn ốc ra đúng 1..n²


# 8. Vì sao đúng? Độ phức tạp?

**Bất biến:** đầu mỗi vòng, các ô **ngoài** hình chữ nhật `[top..bottom] × [left..right]` đã được điền đúng bằng `1..num-1` theo thứ tự xoắn ốc, các ô bên trong còn `0`. Mỗi vòng điền một lớp vỏ theo đúng thứ tự và co biên lại.

```text
Time:  O(n²)   — mỗi ô điền đúng một lần
Space: O(1)    — ngoài ma trận kết quả
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Tạo ma trận bằng `[[0] * n] * n`

In [12]:
def wrong_shared_rows(n: int) -> List[List[int]]:
    mat = [[0] * n] * n                              # SAI
    top, bottom, left, right = 0, n - 1, 0, n - 1
    num = 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):
            mat[top][c] = num; num += 1
        top += 1
        for r in range(top, bottom + 1):
            mat[r][right] = num; num += 1
        right -= 1
        if top <= bottom:
            for c in range(right, left - 1, -1):
                mat[bottom][c] = num; num += 1
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1):
                mat[r][left] = num; num += 1
            left += 1
    return mat

show(wrong_shared_rows(3))

8 9 5
8 9 5
8 9 5


Mọi hàng là cùng một list, nên chỉ còn giá trị **ghi sau cùng** ở mỗi cột.

## Lỗi 2: `range` ngược thiếu `- 1`

`range(right, left, -1)` bỏ sót cột `left` → còn ô `0` trong kết quả.

## Lỗi 3: Quên co biên sau mỗi cạnh

Thiếu `top += 1` sẽ điền đè lên hàng vừa điền ở bước "xuống".

## Lỗi 4: Điều kiện vòng lặp `while num <= n * n` mà không co biên đúng

Có thể dừng đúng số lượng nhưng ghi đè sai chỗ — luôn kiểm tra bằng cách **đọc ngược** xoắn ốc như ở mục 7.

# 10. Mẹo nhận diện pattern

- **Đọc** xoắn ốc (#54) và **ghi** xoắn ốc (#59) dùng **cùng** khung bốn biên — chỉ khác thân vòng lặp.
- Kiểm tra lời giải "ghi" bằng lời giải "đọc" (và ngược lại) là một kỹ thuật test rất hiệu quả.
- Bài liên quan: **#54 Spiral Matrix**, **#885 Spiral Matrix III**, **#2326 Spiral Matrix IV** (điền từ linked list).

## Template

```python
top, bottom, left, right = 0, n - 1, 0, n - 1
while top <= bottom and left <= right:
    ... hàng top, trái -> phải;   top += 1
    ... cột right, trên -> dưới;  right -= 1
    if top <= bottom: ... hàng bottom, phải -> trái; bottom -= 1
    if left <= right: ... cột left, dưới -> trên;    left += 1
```

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Viết ra ma trận cho `n = 5` (không chạy code). Số ở chính giữa là bao nhiêu? Số ở `(1, 1)` là bao nhiêu?

## Bài 2 — Điền chỗ trống

In [13]:
def exercise_2_generate_matrix(n: int) -> List[List[int]]:
    mat = [[0] * n for _ in range(n)]
    top, bottom, left, right = 0, n - 1, 0, n - 1
    num = 1
    while top <= bottom and left <= right:
        # TODO: điền 4 cạnh, co biên sau mỗi cạnh
        break
    return mat


# assert exercise_2_generate_matrix(3) == [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
# assert exercise_2_generate_matrix(1) == [[1]]

## Bài 3 — Biến thể: ma trận chữ nhật `m × n`

```python
generate_rect(3, 4) -> [[1,  2,  3, 4],
                        [10, 11, 12, 5],
                        [9,  8,  7, 6]]
```

Lúc này các kiểm tra `if top <= bottom` / `if left <= right` là **bắt buộc**.

In [14]:
def exercise_3_generate_rect(m: int, n: int) -> List[List[int]]:
    # TODO
    pass


# assert exercise_3_generate_rect(3, 4) == [[1, 2, 3, 4], [10, 11, 12, 5], [9, 8, 7, 6]]
# assert exercise_3_generate_rect(1, 3) == [[1, 2, 3]]
# assert exercise_3_generate_rect(3, 1) == [[1], [2], [3]]

## Bài 4 — Biến thể: xoắn ốc ngược chiều kim đồng hồ

Đi **xuống → phải → lên → trái**:

```python
generate_ccw(3) -> [[1, 8, 7],
                    [2, 9, 6],
                    [3, 4, 5]]
```

Gợi ý nhanh: ma trận này chính là **chuyển vị** của ma trận xoắn ốc thuận chiều.

In [15]:
def exercise_4_generate_ccw(n: int) -> List[List[int]]:
    # TODO
    pass


# assert exercise_4_generate_ccw(3) == [[1, 8, 7], [2, 9, 6], [3, 4, 5]]
# assert exercise_4_generate_ccw(1) == [[1]]

## Bài 5 — Mini challenge: giá trị tại `(r, c)` không cần dựng ma trận

Tính `generateMatrix(n)[r][c]` trong `O(1)` (hoặc `O(số lớp)`), không tạo ma trận.

Gợi ý nhanh: ô `(r, c)` nằm ở lớp `k = min(r, c, n-1-r, n-1-c)`. Lớp `k` bắt đầu bằng số `4k(n-k) + 1` (tổng số ô của `k` lớp ngoài). Trong lớp, xác định ô thuộc cạnh nào.

In [16]:
def exercise_5_value_at(n: int, r: int, c: int) -> int:
    # TODO
    pass


# assert exercise_5_value_at(3, 1, 1) == 9
# assert exercise_5_value_at(4, 2, 1) == 16
# assert all(exercise_5_value_at(n, r, c) == generate_matrix(n)[r][c] for n in range(1, 9) for r in range(n) for c in range(n))

# 12. Gợi ý

**Bài 1**
- Mức 1: lớp 0 có `4·5 - 4 = 16` ô (`1..16`), lớp 1 có `4·3 - 4 = 8` ô (`17..24`), tâm là `25`.

**Bài 3**
- Mức 1: giống lời giải chính, `bottom = m - 1`, `right = n - 1`.

**Bài 4**
- Mức 1: `mat = generate_matrix(n)`; trả về `[list(col) for col in zip(*mat)]`.

**Bài 5**
- Mức 1: `side = n - 2k` là cạnh của lớp `k`; `start = 4k(n - k) + 1`.
- Mức 2: nếu `r == k`: cạnh trên, giá trị `start + (c - k)`.
- Mức 3: nếu `c == n-1-k`: cạnh phải `start + (side-1) + (r-k)`; cạnh dưới `start + 2(side-1) + (n-1-k-c)`; cạnh trái `start + 3(side-1) + (n-1-k-r)`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [17]:
show(generate_matrix(5))
print("chính giữa:", generate_matrix(5)[2][2], "| (1, 1):", generate_matrix(5)[1][1])

 1  2  3  4  5
16 17 18 19  6
15 24 25 20  7
14 23 22 21  8
13 12 11 10  9
chính giữa: 25 | (1, 1): 17


## Đáp án Bài 2

In [18]:
def answer_2_generate_matrix(n: int) -> List[List[int]]:
    return Solution().generateMatrix(n)

assert answer_2_generate_matrix(3) == [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
assert answer_2_generate_matrix(1) == [[1]]
print("Bài 2 PASS (giống lời giải chính mục 7)")

Bài 2 PASS (giống lời giải chính mục 7)


## Đáp án Bài 3

In [19]:
def answer_3_generate_rect(m: int, n: int) -> List[List[int]]:
    mat = [[0] * n for _ in range(m)]
    top, bottom, left, right = 0, m - 1, 0, n - 1
    num = 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):
            mat[top][c] = num; num += 1
        top += 1
        for r in range(top, bottom + 1):
            mat[r][right] = num; num += 1
        right -= 1
        if top <= bottom:
            for c in range(right, left - 1, -1):
                mat[bottom][c] = num; num += 1
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1):
                mat[r][left] = num; num += 1
            left += 1
    return mat

assert answer_3_generate_rect(3, 4) == [[1, 2, 3, 4], [10, 11, 12, 5], [9, 8, 7, 6]]
assert answer_3_generate_rect(1, 3) == [[1, 2, 3]]
assert answer_3_generate_rect(3, 1) == [[1], [2], [3]]
for m_ in range(1, 8):
    for n_ in range(1, 8):
        assert spiral_read(answer_3_generate_rect(m_, n_)) == list(range(1, m_ * n_ + 1))
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [20]:
def answer_4_generate_ccw(n: int) -> List[List[int]]:
    return [list(col) for col in zip(*generate_matrix(n))]

assert answer_4_generate_ccw(3) == [[1, 8, 7], [2, 9, 6], [3, 4, 5]]
assert answer_4_generate_ccw(1) == [[1]]
print("Bài 4 PASS")

Bài 4 PASS


Chuyển vị đổi vai trò hàng ↔ cột, nên "đi phải" thành "đi xuống", "đi xuống" thành "đi phải", … — đúng là xoắn ốc ngược chiều.

## Đáp án Bài 5

In [21]:
def answer_5_value_at(n: int, r: int, c: int) -> int:
    k = min(r, c, n - 1 - r, n - 1 - c)
    side = n - 2 * k
    start = 4 * k * (n - k) + 1
    if side == 1:
        return start
    if r == k:
        return start + (c - k)
    if c == n - 1 - k:
        return start + (side - 1) + (r - k)
    if r == n - 1 - k:
        return start + 2 * (side - 1) + (n - 1 - k - c)
    return start + 3 * (side - 1) + (n - 1 - k - r)

assert answer_5_value_at(3, 1, 1) == 9
assert answer_5_value_at(4, 2, 1) == 16
assert all(answer_5_value_at(n, r, c) == generate_matrix(n)[r][c] for n in range(1, 9) for r in range(n) for c in range(n))
print("Bài 5 PASS (khớp mọi ô với n = 1..8)")

Bài 5 PASS (khớp mọi ô với n = 1..8)


`4k(n - k)` là số ô của `k` lớp ngoài: lớp `i` có `4(n - 2i) - 4` ô, cộng `i = 0..k-1` ra `4k(n - k)`.

# 14. Tổng kết

> Khung bốn biên của bài #54, chỉ thay "đọc" bằng "ghi `num++`". Tạo ma trận bằng list comprehension, co biên sau mỗi cạnh, và kiểm tra bằng cách đọc xoắn ốc ngược lại.

---

## Checklist trước khi nộp

- [ ] Tạo ma trận bằng `[[0] * n for _ in range(n)]`?
- [ ] Co biên sau **mỗi** cạnh?
- [ ] `range` ngược dùng `left - 1` / `top - 1`?
- [ ] Đã thử `n = 1`, `n = 2`, `n` lẻ?